# 문화누리 만족활동 기반 선호분석 — 전체 재실행

이 노트북은 **Run All 한 번으로 전체 파이프라인을 순서대로 재실행**합니다.

1. 활동코드 중분류 매핑 및 만족활동 1·2·3순위 검증
2. 3:2:1 순위가중 다항 로지스틱 학습·검증
3. 100m 격자 성별×연령별 문화누리 대상자 인구 정렬
4. 격자·행정동·자치구 잠재수요, 외적 타당성, HTML 지도 생성
5. 전체 자동테스트

계산 로직을 노트북에 복사하지 않고 검증된 `src/preference_analysis` 모듈을 호출합니다. 원본 데이터는 수정하지 않으며 `data/processed` 산출물만 새로 생성합니다. 접근성·가맹점·이동시간 변수는 선호모델에 사용하지 않습니다.

> 전체 모델 학습과 60,528개 격자 지도 생성 때문에 실행에 시간이 걸릴 수 있습니다. 실행 중인 셀 왼쪽이 `[*]`이면 정상적으로 계산 중입니다.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import FileLink, Markdown, display

def find_project_root():
    configured = os.environ.get('ORACLE_PROJECT_ROOT')
    starts = ([Path(configured).expanduser()] if configured else []) + [Path.cwd()]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / '.git').exists() and (candidate / 'data').is_dir():
                return candidate.resolve()
    raise FileNotFoundError('Oracle-Project 저장소를 찾지 못했습니다.')

ROOT = find_project_root()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_DIR = ROOT / 'data/processed/preference_analysis/run_logs' / RUN_ID
LOG_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# 전체 실행 기본값입니다. 필요할 때만 False로 바꾸세요.
BUILD_MAPS = True
RUN_EXTERNAL_VALIDATION = True
RUN_TESTS = True

def run_step(step_name, module, *module_args):
    command = [sys.executable, '-m', module, *map(str, module_args)]
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(ROOT), str(ROOT / 'src')])
    print(f'▶ {step_name}')
    print('  ', ' '.join(command))
    started = time.perf_counter()
    completed = subprocess.run(
        command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace'
    )
    elapsed = time.perf_counter() - started
    log_path = LOG_DIR / f'{step_name}.log'
    log_path.write_text(completed.stdout, encoding='utf-8')
    tail = '\n'.join(completed.stdout.splitlines()[-20:])
    if completed.returncode != 0:
        print(tail)
        raise RuntimeError(f'{step_name} 실패 — 전체 로그: {log_path}')
    print(f'✓ 완료 ({elapsed:.1f}초) — 로그: {log_path}')
    if tail:
        print(tail)
    return log_path

print('프로젝트:', ROOT)
print('Python:', sys.executable)
print('실행 로그:', LOG_DIR)

프로젝트: .
Python: ./.venv/bin/python
실행 로그: ./data/processed/preference_analysis/run_logs/20260811_093358


## 1단계 — 중분류 매핑과 만족활동 순위 검증

국민여가활동조사의 활동코드 1~88을 정책 8개 분야와 `기타·문화누리 비대응`으로 변환합니다. 음악 활동은 원본 라벨을 보존하되 정책 출력에서는 기타로 통합하며, 향후 희망활동은 사용하지 않습니다.

In [2]:
run_step(
    '01_중분류_매핑_검증',
    'src.preference_analysis.build_mapping',
    '--project-root', ROOT,
)

▶ 01_중분류_매핑_검증
   ./.venv/bin/python -m src.preference_analysis.build_mapping --project-root .


✓ 완료 (12.6초) — 로그: ./data/processed/preference_analysis/run_logs/20260811_093358/01_중분류_매핑_검증.log
mapping: ./data/processed/preference_analysis/leisure_activity_middle_category_mapping.csv (88 rows)
transformed: ./data/processed/preference_analysis/satisfaction_rank_middle_category_2021_2025.csv (50,238 rows)
validation: ./data/processed/preference_analysis/mapping_validation_summary.csv
distribution: ./data/processed/preference_analysis/satisfaction_mapping_distribution.csv
                                       check  status  value                                                   detail
              mapping_activity_code_coverage    pass     88                             매핑표가 원본 활동코드 1~88을 정확히 포함하는지
                       source_rows_preserved    pass  50238                                            변환 전후 응답자 행 수
               original_rank_code_mismatches    pass      0                             만족활동 1~3순위 원본코드 왕복 비교 불일치 건수
                unmapped_satisfaction_values    pass

PosixPath('./data/processed/preference_analysis/run_logs/20260811_093358/01_중분류_매핑_검증.log')

## 2단계 — 순위가중 다항 로지스틱 학습·검증

만족활동 1·2·3순위에 3:2:1 가중치를 적용합니다. 2022~2024 순차검증과 2021~2024 응답자 그룹 5-Fold Log Loss로 성별×연령 결합형과 C를 선택한 뒤, 2025년을 시간 외 평가자료로 사용합니다.

In [3]:
run_step(
    '02_선호모델_학습_검증',
    'src.preference_analysis.train_model',
    '--project-root', ROOT,
)

▶ 02_선호모델_학습_검증
   ./.venv/bin/python -m src.preference_analysis.train_model --project-root .


✓ 완료 (578.4초) — 로그: ./data/processed/preference_analysis/run_logs/20260811_093358/02_선호모델_학습_검증.log
            2025 weighted_prior_baseline  0.419421       0.776691  1.607208       20.044649                   20.044649                          0.000000          0.742941                 93.977866                1.338252

outputs
./data/processed/preference_analysis/model/multinomial_tuning_2024.csv
./data/processed/preference_analysis/model/multinomial_tuning_temporal_cv_2022_2024.csv
./data/processed/preference_analysis/model/multinomial_tuning_temporal_cv_by_fold_2022_2024.csv
./data/processed/preference_analysis/model/multinomial_tuning_grouped_cv_2021_2024.csv
./data/processed/preference_analysis/model/multinomial_tuning_grouped_cv_by_fold_2021_2024.csv
./data/processed/preference_analysis/model/multinomial_tuning_combined_cv.csv
./data/processed/preference_analysis/model/model_score_temporal_2022_2025.csv
./data/processed/preference_analysis/model/model_score_2024_2025.csv
./data/

PosixPath('./data/processed/preference_analysis/run_logs/20260811_093358/02_선호모델_학습_검증.log')

## 3단계 — 100m 격자 성별×연령별 대상자 인구 정렬

기존 격자 대상자 추정치를 모델의 7개 연령구간과 성별코드에 맞춥니다. 15세 미만은 현재 선호모델 적용 대상에서 제외됩니다.

In [4]:
run_step(
    '03_격자_성연령_인구정렬',
    'src.preference_analysis.align_population',
    '--project-root', ROOT,
)

▶ 03_격자_성연령_인구정렬
   ./.venv/bin/python -m src.preference_analysis.align_population --project-root .


✓ 완료 (17.3초) — 로그: ./data/processed/preference_analysis/run_logs/20260811_093358/03_격자_성연령_인구정렬.log
                 7            60-69세               6             60대          선호모형_적용                         선호확률 결합 대상          121056                  90175
                 8            70세 이상               7          70대 이상          선호모형_적용                  70세 이상 선호확률 결합 대상          121056                  78572

검증 결과
                             metric   value        expected status                  note
                        source_rows 1452672         1452672   pass          원본 성별×연령 행 수
                  source_grid_count   60528           60528   pass       원본 고유 100m 격자 수
            source_population_total  582549          582549   pass           통일 전 대상자 총량
           aligned_population_total  582549          582549   pass           통일 후 대상자 총량
      population_conservation_error       0               0   pass       통일 후 총량-통일 전 총량
       model_ready_population_total  54

PosixPath('./data/processed/preference_analysis/run_logs/20260811_093358/03_격자_성연령_인구정렬.log')

## 4단계 — 격자·행정동·자치구 잠재수요와 지도

각 성별×연령 셀의 대상자 수에 분야별 절대 선호확률을 곱해 100m 잠재수요를 계산하고 행정동·자치구로 보존 집계합니다. 기본값은 외적 타당성 점검과 HTML 지도까지 생성합니다.

In [5]:
spatial_args = ['--project-root', ROOT]
if not BUILD_MAPS:
    spatial_args.append('--skip-maps')
if not RUN_EXTERNAL_VALIDATION:
    spatial_args.append('--skip-external-validation')
run_step(
    '04_공간결과_외부검증_지도생성',
    'src.preference_analysis.build_spatial_outputs',
    *spatial_args,
)

▶ 04_공간결과_외부검증_지도생성
   ./.venv/bin/python -m src.preference_analysis.build_spatial_outputs --project-root .


✓ 완료 (26.8초) — 로그: ./data/processed/preference_analysis/run_logs/20260811_093358/04_공간결과_외부검증_지도생성.log
       dong_potential_demand_conservation   pass  0.000000e+00      격자→dong 정책·기타 잠재수요 최대 총량 오차
                   gu_target_conservation   pass  0.000000e+00                  격자→gu 대상자 총량 오차
         gu_potential_demand_conservation   pass  5.820766e-11        격자→gu 정책·기타 잠재수요 최대 총량 오차
outputs
./data/processed/preference_analysis/spatial/grid_middle_category_preference_demand_2024.csv
./data/processed/preference_analysis/spatial/dong_middle_category_preference_demand_2024.csv
./data/processed/preference_analysis/spatial/gu_middle_category_preference_demand_2024.csv
./data/processed/preference_analysis/spatial/spatial_validation_summary_2024.csv
./data/processed/preference_analysis/spatial/external_validation_2024_by_gu_category.csv
./data/processed/preference_analysis/spatial/external_validation_2024_by_gu.csv
./data/processed/preference_analysis/spatial/external_validation_2024_by_c

PosixPath('./data/processed/preference_analysis/run_logs/20260811_093358/04_공간결과_외부검증_지도생성.log')

## 5단계 — 전체 자동테스트

매핑, 모델 누수·확률합, 인구 총량, 격자→동→구 보존, 무자료 처리와 지도 생성을 자동검증합니다.

In [6]:
if RUN_TESTS:
    run_step(
        '05_전체_자동테스트',
        'pytest',
        'tests/preference_analysis', '-q',
    )
else:
    print('RUN_TESTS=False — 자동테스트를 건너뛰었습니다.')

▶ 05_전체_자동테스트
   ./.venv/bin/python -m pytest tests/preference_analysis -q


✓ 완료 (9.7초) — 로그: ./data/processed/preference_analysis/run_logs/20260811_093358/05_전체_자동테스트.log
.....................................................                [100%]
53 passed, 4 subtests passed in 9.16s


## 최종 핵심 결과

아래 셀은 전체 계산 후 핵심 성능, 공간 보존검증, 결과 규모와 지도 링크만 간결하게 보여줍니다.

In [7]:
MODEL_DIR = ROOT / 'data/processed/preference_analysis/model'
SPATIAL_DIR = ROOT / 'data/processed/preference_analysis/spatial'

scores = pd.read_csv(MODEL_DIR / 'model_score_2024_2025.csv', encoding='utf-8-sig')
score_columns = [
    'evaluation_year', 'model', 'feature_mode', 'c_value',
    'accuracy', 'top3_accuracy', 'log_loss', 'multiclass_brier',
    'log_loss_skill_score_vs_baseline',
]
display(Markdown('### 모델 성능'))
display(scores[score_columns].round(4))

spatial_validation = pd.read_csv(
    SPATIAL_DIR / 'spatial_validation_summary_2024.csv', encoding='utf-8-sig'
)
display(Markdown('### 공간 계산 검증'))
display(spatial_validation)

metadata = json.loads(
    (SPATIAL_DIR / 'spatial_preference_run_metadata_2024.json').read_text(encoding='utf-8')
)
result_scale = pd.DataFrame([{
    '100m 격자 수': metadata['grid_count'],
    '행정동 수': metadata['dong_count'],
    '자치구 수': metadata['gu_count'],
    '15세 이상 추정 대상자': metadata['target_population_total_15plus'],
    '정책 8개 분야 잠재수요 합': metadata['policy_potential_demand_total'],
    '기타 잠재수요 합': metadata['other_potential_demand_total'],
}])
display(Markdown('### 공간 결과 규모'))
display(result_scale.round(2))

grid_map = SPATIAL_DIR / 'maps/grid_preference_demand_2024.html'
dong_map = SPATIAL_DIR / 'maps/dong_preference_demand_2024.html'
display(Markdown('### 지도 열기'))
if grid_map.exists():
    display(FileLink(str(grid_map), result_html_prefix='100m 격자 지도: '))
if dong_map.exists():
    display(FileLink(str(dong_map), result_html_prefix='행정동 지도: '))

print('✓ 전체 파이프라인 완료')
print('실행 로그:', LOG_DIR)

### 모델 성능

,evaluation_year,model,feature_mode,c_value,accuracy,top3_accuracy,log_loss,multiclass_brier,log_loss_skill_score_vs_baseline
0,2024,multinomial_logistic,sex_age_interaction,0.1,0.4266,0.8019,1.5368,0.7262,2.3333
1,2024,weighted_prior_baseline,weighted_prior,NaN,0.4266,0.7873,1.5735,0.7372,0.0000
2,2025,multinomial_logistic,sex_age_interaction,0.1,0.4194,0.7860,1.5695,0.7359,2.3431
3,2025,weighted_prior_baseline,weighted_prior,NaN,0.4194,0.7767,1.6072,0.7429,0.0000


### 공간 계산 검증

,check,status,value,detail
0,grid_policy_plus_other_equals_target,pass,-2.328306e-10,정책 8개 잠재수요와 기타 잠재수요 합의 대상자 총량 오차
1,grid_category_key_unique,pass,0.000000e+00,GRID_CD×분야 중복 행 수
2,zero_target_grids_are_no_data,pass,3.211900e+04,대상자 0명 격자는 확률 무자료·잠재수요 0으로 보존
3,grid_conditional_policy_share_sums_to_one,pass,4.440892e-16,대상자 양수 격자의 정책 8개 조건부 구성비 최대 절대오차
4,dong_target_conservation,pass,0.000000e+00,격자→dong 대상자 총량 오차
5,dong_potential_demand_conservation,pass,0.000000e+00,격자→dong 정책·기타 잠재수요 최대 총량 오차
6,gu_target_conservation,pass,0.000000e+00,격자→gu 대상자 총량 오차
7,gu_potential_demand_conservation,pass,5.820766e-11,격자→gu 정책·기타 잠재수요 최대 총량 오차


### 공간 결과 규모

,100m 격자 수,행정동 수,자치구 수,15세 이상 추정 대상자,정책 8개 분야 잠재수요 합,기타 잠재수요 합
0,60528,426,25,545692.0,317092.02,228599.98


### 지도 열기

./data/processed/preference_analysis/spatial/maps/grid_preference_demand_2024.html

./data/processed/preference_analysis/spatial/maps/dong_preference_demand_2024.html

✓ 전체 파이프라인 완료
실행 로그: ./data/processed/preference_analysis/run_logs/20260811_093358
